# 🚀 HUẾ SMART TOURISM - GPU BACKEND SERVER TRÊN GOOGLE COLAB
**Đề tài Khóa luận Tốt nghiệp:** Hệ thống Hỏi đáp và Gợi ý Lịch trình Du lịch Thông minh dựa trên Spatial GraphRAG & LLM
**Nhánh phát triển:** `quylhm`
---
Notebook này cho phép bạn chạy toàn bộ **FastAPI Backend + Ollama (Qwen2.5) + ChromaDB + Spatial Knowledge Graph** hoàn toàn miễn phí trên GPU **Nvidia T4** của Google Colab.

### ⚡ BẢN TỐI ƯU TỐC ĐỘ (RÚT NGẮN TỪ 5 PHÚT XUỐNG 30-45s):
- **Bước 4:** Dùng `uv` cài thư viện trong **10s** (thay vì 2.5 phút của pip).
- **Bước 5:** Lưu vĩnh viễn Ollama model vào Google Drive (`KhoaLuan_Data/ollama_models`) -> **0s tải lại** 2GB model từ lần thứ 2!
- **Bước 7:** Lưu vĩnh viễn HuggingFace embeddings vào Google Drive (`KhoaLuan_Data/hf_cache`) -> **0s tải lại** 1.1GB weights!

### ⚠️ BƯỚC 0: BẬT GPU TRƯỚC KHI CHẠY (BẮT BUỘC)
1. Trên thanh menu Colab, chọn **Runtime** (Thời gian chạy) -> **Change runtime type** (Thay đổi loại thời gian chạy).
2. Tại mục **Hardware accelerator** (Bộ tăng tốc phần cứng), chọn **T4 GPU**.
3. Bấm **Save** (Lưu).

### BƯỚC 1: KIỂM TRA GPU NVIDIA T4

In [ ]:
!nvidia-smi

### BƯỚC 2: KẾT NỐI GOOGLE DRIVE (ĐỂ LẤY DỮ LIỆU CHROMADB & MODEL CACHE)
> Thư mục `KhoaLuan_Data` trên Google Drive sẽ chứa `chroma_db` và tự động lưu `ollama_models`, `hf_cache` vĩnh viễn để không bao giờ phải tải lại.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Tạo sẵn các thư mục cache trên Drive để lưu trữ vĩnh viễn
DRIVE_DATA_DIR = "/content/drive/MyDrive/KhoaLuan_Data"
os.makedirs(f"{DRIVE_DATA_DIR}/ollama_models", exist_ok=True)
os.makedirs(f"{DRIVE_DATA_DIR}/hf_cache", exist_ok=True)

print("✅ Đã kết nối Google Drive & chuẩn bị thư mục cache thành công!")

### BƯỚC 3: CLONE MÃ NGUỒN KHÓA LUẬN (NHÁNH QUYLHM) VÀ LIÊN KẾT DỮ LIỆU

In [ ]:
# 1. Clone repository nhánh quylhm nếu chưa có
%cd /content
if not os.path.exists('/content/KhoaLuanTotNghiep'):
    !git clone -b quylhm https://github.com/minhquy017/KhoaLuanTotNghiep.git
else:
    %cd /content/KhoaLuanTotNghiep
    !git checkout quylhm
    !git pull origin quylhm

%cd /content/KhoaLuanTotNghiep

# 2. Tạo liên kết (symlink) thư mục chroma_db từ Google Drive vào project
# Đường dẫn mẫu trên Drive: /content/drive/MyDrive/KhoaLuan_Data/chroma_db
drive_chroma_path = "/content/drive/MyDrive/KhoaLuan_Data/chroma_db"
local_chroma_path = "/content/KhoaLuanTotNghiep/data/processed/chroma_db"

if os.path.exists(drive_chroma_path):
    if os.path.exists(local_chroma_path) or os.path.islink(local_chroma_path):
        !rm -rf {local_chroma_path}
    !ln -s "{drive_chroma_path}" "{local_chroma_path}"
    print(f"✅ Đã liên kết ChromaDB từ Drive vào: {local_chroma_path}")
else:
    print(f"⚠️ Không tìm thấy ChromaDB tại '{drive_chroma_path}'. Hãy kiểm tra lại đường dẫn thư mục trên Google Drive của bạn!")

### BƯỚC 4: CÀI ĐẶT CÁC THƯ VIỆN PYTHON CẦN THIẾT (TỐI ƯU VỚI UV ~10 GIÂY THAY VÌ 2.5 PHÚT)

In [ ]:
# Cài uv (Rust-based) để tải và giải nén song song siêu tốc (~10s thay vì 2.5 phút)
!pip install -q uv
!uv pip install --system -q fastapi uvicorn langgraph langchain-core langchain-ollama chromadb sentence-transformers networkx duckduckgo-search pydantic python-dotenv
print("✅ Đã cài đặt xong toàn bộ thư viện cần thiết!")

### BƯỚC 5: CÀI ĐẶT OLLAMA VÀ LƯU CACHE MÔ HÌNH VÀO GOOGLE DRIVE (TỪ LẦN THỨ 2: 0s TẢI LẠI)

In [ ]:
import subprocess
import time
import os

# 1. Cài đặt Ollama cho Linux (nếu chưa cài)
if not os.path.exists("/usr/local/bin/ollama"):
    print("📥 Đang cài đặt Ollama...")
    !curl -fsSL https://ollama.com/install.sh | sh

# 2. Liên kết thư mục model của Ollama vào Google Drive để không bao giờ bị mất
!mkdir -p /root/.ollama
drive_ollama_path = "/content/drive/MyDrive/KhoaLuan_Data/ollama_models"
if not os.path.exists("/root/.ollama/models") and not os.path.islink("/root/.ollama/models"):
    !ln -s "{drive_ollama_path}" /root/.ollama/models

# 3. Khởi chạy tiến trình Ollama daemon ngầm
print("⚡ Đang khởi động Ollama daemon...")
ollama_env = os.environ.copy()
ollama_env["OLLAMA_MODELS"] = drive_ollama_path
ollama_process = subprocess.Popen(["ollama", "serve"], env=ollama_env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

# 4. Tải mô hình Qwen2.5 (Nếu đã có trong Drive thì chỉ mất 0.5s kiểm tra, không tải lại!)
# 'qwen2.5:3b'  : Khuyên dùng cho QA văn bản & gợi ý lịch trình (nhẹ, cực nhanh)
# 'qwen2.5vl:7b': Dành cho nhận diện ảnh thị giác từ du khách
MODEL_NAME = "qwen2.5:3b"
print(f"🔥 Đang kiểm tra mô hình {MODEL_NAME}...")
!ollama pull {MODEL_NAME}

print(f"✅ Ollama và {MODEL_NAME} đã sẵn sàng trên GPU T4!")

### BƯỚC 6: CÀI ĐẶT CLOUDFLARE TUNNEL (ĐƯỜNG HẦM CÔNG KHAI HTTPS MIỄN PHÍ)

In [ ]:
# Tải trực tiếp binary cloudflared (chỉ mất ~2s)
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
print("✅ Đã cài đặt Cloudflare Tunnel thành công!")

### BƯỚC 7: KHỞI CHẠY BACKEND FASTAPI & MỞ ĐƯỜNG LINK PUBLIC
> Chạy cell này để khởi động Server. Đường link HTTPS công khai sẽ được in ra ở cuối. Bạn chỉ việc copy link đó dán vào `frontend/src/api.js`!

In [ ]:
import subprocess
import time
import re
import os

%cd /content/KhoaLuanTotNghiep

# Tối ưu: Cache HuggingFace Embeddings (MiniLM + CLIP) trên Drive để không phải tải lại 1.1GB
os.environ["HF_HOME"] = "/content/drive/MyDrive/KhoaLuan_Data/hf_cache"
os.environ["OLLAMA_MODEL"] = "qwen2.5:3b"

# 1. Khởi chạy FastAPI Backend trên cổng 8000
print("🚀 Đang khởi động FastAPI Backend (port 8000)...")
backend_log = open("/content/backend.log", "w")
server_process = subprocess.Popen(
    ["uvicorn", "src.app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    env=os.environ.copy(),
    stdout=backend_log,
    stderr=subprocess.STDOUT
)
time.sleep(4)

# 2. Kích hoạt Cloudflare Tunnel trỏ tới port 8000
print("🌐 Đang thiết lập đường hầm Cloudflare Tunnel...")
tunnel_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
start_time = time.time()
while time.time() - start_time < 30:
    line = tunnel_process.stdout.readline()
    if not line:
        continue
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

print("\n" + "="*70)
if public_url:
    print(f"🎉 BACKEND SERVER ĐÃ CHẠY THÀNH CÔNG TRÊN GPU T4!")
    print(f"🔗 PUBLIC API URL: {public_url}")
    print(f"📚 API DOCS (SWAGGER): {public_url}/docs")
    print("="*70)
    print("\n👉 BƯỚC TIẾP THEO Ở MÁY CỦA BẠN:")
    print(f"Mở file `frontend/src/api.js`, đổi dòng số 3 thành:")
    print(f"   const API_BASE_URL = '{public_url}';")
    print("Và trải nghiệm ứng dụng Web bình thường, máy tính không còn tốn 1% GPU nào!")
else:
    print("⚠️ Không tìm thấy URL Cloudflare trong 30 giây. Xem log: !cat /content/backend.log")

# Giữ tiến trình sống để Colab không dừng
try:
    while True:
        time.sleep(10)
except KeyboardInterrupt:
    print("🛑 Đã dừng server.")

---
## ⚡ TÙY CHỌN BỔ SUNG: 1-CLICK TURBO LAUNCHER (CHẠY TẤT CẢ 7 BƯỚC TRONG 1 CELL DUY NHẤT)
> Nếu sau này bạn không muốn bấm lần lượt từng cell từ 1 đến 7 nữa, bạn có thể chạy cell gộp dưới đây để nó tự động hoàn tất từ A-Z trong ~35-45s!

In [ ]:
import os, sys, time, subprocess, re
from google.colab import drive

MODEL_NAME = "qwen2.5:3b"

# 1. Mount Drive & chuẩn bị thư mục cache
drive.mount('/content/drive')
DRIVE_BASE = "/content/drive/MyDrive/KhoaLuan_Data"
for f in ["ollama_models", "hf_cache"]:
    os.makedirs(f"{DRIVE_BASE}/{f}", exist_ok=True)

# 2. Cài đặt thư viện siêu tốc với uv (~10s)
!pip install -q uv
!uv pip install --system -q fastapi uvicorn langgraph langchain-core langchain-ollama chromadb sentence-transformers networkx duckduckgo-search pydantic python-dotenv

# 3. Clone / Pull nhánh quylhm & symlink ChromaDB
%cd /content
if not os.path.exists('/content/KhoaLuanTotNghiep'):
    !git clone -b quylhm https://github.com/minhquy017/KhoaLuanTotNghiep.git
else:
    %cd /content/KhoaLuanTotNghiep
    !git checkout quylhm
    !git pull origin quylhm

%cd /content/KhoaLuanTotNghiep
local_chroma = "/content/KhoaLuanTotNghiep/data/processed/chroma_db"
if os.path.exists(f"{DRIVE_BASE}/chroma_db"):
    if os.path.exists(local_chroma) or os.path.islink(local_chroma):
        !rm -rf {local_chroma}
    !ln -s "{DRIVE_BASE}/chroma_db" "{local_chroma}"

# 4. Cài đặt Ollama daemon & trỏ cache model vào Drive
if not os.path.exists("/usr/local/bin/ollama"):
    !curl -fsSL https://ollama.com/install.sh | sh
!mkdir -p /root/.ollama
if not os.path.exists("/root/.ollama/models") and not os.path.islink("/root/.ollama/models"):
    !ln -s "{DRIVE_BASE}/ollama_models" /root/.ollama/models

ollama_env = os.environ.copy()
ollama_env["OLLAMA_MODELS"] = f"{DRIVE_BASE}/ollama_models"
subprocess.Popen(["ollama", "serve"], env=ollama_env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
!ollama pull {MODEL_NAME}

# 5. Cài đặt Cloudflare Tunnel
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

# 6. Khởi chạy FastAPI Backend & Lấy public link
os.environ["HF_HOME"] = f"{DRIVE_BASE}/hf_cache"
os.environ["OLLAMA_MODEL"] = MODEL_NAME

backend_log = open("/content/backend.log", "w")
server_process = subprocess.Popen(
    ["uvicorn", "src.app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    env=os.environ.copy(),
    stdout=backend_log,
    stderr=subprocess.STDOUT
)
time.sleep(3)

tunnel_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
start_time = time.time()
while time.time() - start_time < 35:
    line = tunnel_process.stdout.readline()
    if not line:
        continue
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

print("\n" + "="*70)
if public_url:
    print(f"🎉 BACKEND SERVER ĐÃ CHẠY THÀNH CÔNG TRÊN GPU T4!")
    print(f"🔗 PUBLIC API URL: {public_url}")
    print(f"📚 API DOCS (SWAGGER): {public_url}/docs")
    print("="*70)
    print("\n👉 BƯỚC TIẾP THEO Ở MÁY CỦA BẠN:")
    print(f"Mở file `frontend/src/api.js`, đổi dòng số 3 thành:")
    print(f"   const API_BASE_URL = '{public_url}';")
else:
    print("⚠️ Không tìm thấy URL Cloudflare. Xem log: !cat /content/backend.log")

try:
    while True:
        time.sleep(10)
except KeyboardInterrupt:
    print("🛑 Đã dừng server.")
